# CLIP4CAD-GFA v4.9 Training

**Direct Contrastive Alignment (No Codebook)**

## Why v4.8.x Failed

The codebook acts as an information bottleneck:
- 137,000 unique CAD models → 1040 codes → ~15 active per sample → weighted_sum → nearly identical z
- All pairs have ~0.985 cosine, but R@1 = 0%
- The model cannot distinguish sample A from sample B

## v4.9 Solution

**No codebook. No gap-closing. Just strong encoders + InfoNCE.**

```
Encoder → AttentionPooling → Projection → InfoNCE
```

## Architecture
| Component | Description |
|-----------|-------------|
| BRep Encoder | TopologyBRepEncoder with EdgeMessageLayer |
| PC Encoder | 2-layer MLP (1024→d) |
| Text Encoder | proj(3072→d) + 2-layer transformer |
| Pooling | AttentionPooling with K=8 learnable queries |
| Projection | Separate heads per modality |

## Training Stages
| Stage | Epochs | LR | Goal |
|-------|--------|-----|------|
| 0 | 8 | 3e-4 | BRep-PC margin > 0.3 |
| 1 | 20 | 1e-4 | R@1 > 20% by ep10, > 50% by ep20 |
| 2 | 5 | 2e-5 | Fine-tuning with hard negatives |

## Key Metric: MARGIN
```
margin = mean_positive_similarity - mean_negative_similarity

Healthy training:
  Epoch 1:  margin = 0.02  (barely above random)
  Epoch 5:  margin = 0.15  (learning!)
  Epoch 10: margin = 0.30  (good separation)
  Epoch 20: margin = 0.50+ (strong discrimination)

ALARM: margin stays near 0 for >3 epochs → model collapsed
```

In [1]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# Clean memory
gc.collect()
torch.cuda.empty_cache()

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4090
Memory: 25.8 GB


In [2]:
# Cell 2: Configuration
# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS
# ═══════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"
ALIGNED_DIR = DATA_ROOT / "aligned"

# PC and Text files
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

# Output
OUTPUT_DIR = Path("../outputs/gfa_v4_9")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS (v4.9)
# ═══════════════════════════════════════════════════════════════════════════════
BATCH_SIZE = 512  # Larger batch for better contrastive learning

# Stage epochs
STAGE0_EPOCHS = 8     # Anchor BRep to PC
STAGE1_EPOCHS = 20    # Full 3-way alignment
STAGE2_EPOCHS = 5     # Fine-tuning with hard negatives (optional)
TOTAL_EPOCHS = STAGE0_EPOCHS + STAGE1_EPOCHS + STAGE2_EPOCHS

# Learning rates per stage
STAGE0_LR = 3e-4
STAGE1_LR = 1e-4
STAGE2_LR = 2e-5

# Warmup epochs per stage
STAGE0_WARMUP = 1
STAGE1_WARMUP = 2
STAGE2_WARMUP = 0

WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

print("Configuration (v4.9 - No Codebook):")
print(f"  Data root: {DATA_ROOT}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Stage 0: {STAGE0_EPOCHS} epochs, LR={STAGE0_LR}, warmup={STAGE0_WARMUP}")
print(f"  Stage 1: {STAGE1_EPOCHS} epochs, LR={STAGE1_LR}, warmup={STAGE1_WARMUP}")
print(f"  Stage 2: {STAGE2_EPOCHS} epochs, LR={STAGE2_LR}, warmup={STAGE2_WARMUP}")
print(f"  Total: {TOTAL_EPOCHS} epochs")
print(f"  Output: {OUTPUT_DIR}")

Configuration (v4.9 - No Codebook):
  Data root: d:\Defect_Det\MMCAD\data
  Batch size: 512
  Stage 0: 8 epochs, LR=0.0003, warmup=1
  Stage 1: 20 epochs, LR=0.0001, warmup=2
  Stage 2: 5 epochs, LR=2e-05, warmup=0
  Total: 33 epochs
  Output: ..\outputs\gfa_v4_9


In [3]:
# Cell 3: Import Dataset
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn

print("Using GFAMappedDataset with use_autobrep=True")

Using GFAMappedDataset with use_autobrep=True


In [4]:
# Cell 4: Load Datasets
gc.collect()
torch.cuda.empty_cache()

print("Loading datasets with AutoBrep topology...")
print("=" * 60)

MAX_TRAIN_SAMPLES = 111000

# Train dataset - LOAD TO RAM
print(f"\n[1/2] Loading TRAIN dataset (max {MAX_TRAIN_SAMPLES:,} samples)...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=True,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
    max_samples=MAX_TRAIN_SAMPLES,
)
print(f"Train: {len(train_dataset):,} samples")

# Val dataset - ON DISK
print("\n[2/2] Loading VAL dataset...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=False,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
)
print(f"Val: {len(val_dataset):,} samples")

print("\n" + "=" * 60)
print("DATASETS READY!")

Loading datasets with AutoBrep topology...

[1/2] Loading TRAIN dataset (max 111,000 samples)...
  Filtered to 132324 samples with AutoBrep data (from 133105)
  Limiting to 111000 samples (from 132324)
  Loading train data to memory (B-Rep + PC + Text)...
    Loading AutoBrep with topology...
    Loading 111000 samples from train_brep_autobrep.h5...
      Done (111000 samples)
    AutoBrep loaded: 9.2GB in 107.6s
    Loading PC (50GB)...
    Loading Text from: c:\Users\User\Desktop\text_splits\train_text_embeddings.h5
    ✓ Text loaded: 174.6GB in 357.2s
  ✓ Loaded 111000 samples: 205.6GB in RAM (B-Rep + PC + Text)
GFAMappedDataset: train with 111000 samples (in memory)
Train: 111,000 samples

[2/2] Loading VAL dataset...
  Filtered to 16535 samples with AutoBrep data (from 16638)
GFAMappedDataset: val with 16535 samples
Val: 16,535 samples

DATASETS READY!


In [5]:
# Cell 5: Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
)

STEPS_PER_EPOCH = len(train_loader)
print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")

Train loader: 216 batches
Val loader: 33 batches


In [6]:
# Cell 6: Batch Remapping Function
def remap_batch(batch):
    """Remap collate output keys to model expected keys."""
    remapped = {}
    for k, v in batch.items():
        # Remove 'brep_' prefix for model compatibility
        if k.startswith('brep_'):
            new_key = k[5:]  # Remove 'brep_' prefix
            remapped[new_key] = v
        else:
            remapped[k] = v
    
    # Split pc_features into local and global
    if 'pc_features' in remapped:
        pc = remapped.pop('pc_features')
        remapped['pc_local_features'] = pc[:, :-1, :]   # (B, N, 1024)
        remapped['pc_global_features'] = pc[:, -1, :]   # (B, 1024)
    
    # Rename text keys
    if 'desc_embedding' in remapped:
        remapped['text_features'] = remapped.pop('desc_embedding')
    if 'desc_mask' in remapped:
        remapped['text_mask'] = remapped.pop('desc_mask')
    
    return remapped

# Test remapping
test_batch = next(iter(train_loader))
test_batch = remap_batch(test_batch)
print("Remapped batch keys:")
for k, v in test_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

# Verify topology fields
assert 'edge_to_faces' in test_batch, "edge_to_faces required!"
assert 'bfs_level' in test_batch, "bfs_level required!"
print("\nTopology fields verified!")

Remapped batch keys:
  has_brep: torch.Size([512])
  has_pointcloud: torch.Size([512])
  has_text: torch.Size([512])
  face_features: torch.Size([512, 192, 48])
  edge_features: torch.Size([512, 512, 12])
  face_mask: torch.Size([512, 192])
  edge_mask: torch.Size([512, 512])
  edge_to_faces: torch.Size([512, 512, 2])
  bfs_level: torch.Size([512, 192])
  face_centroids: torch.Size([512, 192, 3])
  face_normals: torch.Size([512, 192, 3])
  face_areas: torch.Size([512, 192])
  edge_midpoints: torch.Size([512, 512, 3])
  edge_lengths: torch.Size([512, 512])
  idx: torch.Size([512])
  rot_idx: torch.Size([512])
  pc_local_features: torch.Size([512, 47, 1024])
  pc_global_features: torch.Size([512, 1024])
  text_features: torch.Size([512, 256, 3072])
  text_mask: torch.Size([512, 256])

Topology fields verified!


In [11]:
# Cell 7: Create Model (v4.9 - No Codebook)
gc.collect()
torch.cuda.empty_cache()
import importlib
import clip4cad.models.clip4cad_gfa_v4_9 as v49_module
importlib.reload(v49_module)

from clip4cad.models.clip4cad_gfa_v4_9 import CLIP4CAD_GFA_v49, GFAv49Config

from clip4cad.models.clip4cad_gfa_v4_9 import CLIP4CAD_GFA_v49, GFAv49Config
from clip4cad.losses.gfa_v4_9_losses import (
    GFAv49Loss,
    compute_retrieval_metrics,
    compute_contrastive_quality,
    mine_hard_negatives_simple,
    get_warmup_cosine_scheduler,
)

# Model configuration
config = GFAv49Config(
    # Input dimensions
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    # Model dimensions
    d=256,
    d_proj=128,
    # BRep encoder
    num_msg_layers=3,
    num_brep_tf_layers=4,
    num_heads=8,
    dropout=0.1,
    # Text encoder
    num_text_tf_layers=2,
    # Attention pooling (replaces codebook)
    num_pool_queries=8,
    # Contrastive temperature
    init_tau=0.07,
)




model = CLIP4CAD_GFA_v49(config).to(device)

# Enable gradient checkpointing for memory efficiency
model.enable_gradient_checkpointing()

print(f"\nModel: CLIP4CAD_GFA_v49 (No Codebook)")
print(f"Model parameters: {model.count_parameters():,}")
print(f"Trainable parameters: {model.count_parameters(trainable_only=True):,}")
print(f"\nConfig summary:")
print(f"  d={config.d}, d_proj={config.d_proj}")
print(f"  Attention pooling queries: {config.num_pool_queries}")
print(f"  BRep TF layers: {config.num_brep_tf_layers}, heads: {config.num_heads}")

c:\Users\User\anaconda3\envs\clip4cad\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Enabling gradient checkpointing...
  - BRep transformer: checkpointed
  - Text transformer: checkpointed
Gradient checkpointing enabled

Model: CLIP4CAD_GFA_v49 (No Codebook)
Model parameters: 12,101,249
Trainable parameters: 12,101,249

Config summary:
  d=256, d_proj=128
  Attention pooling queries: 8
  BRep TF layers: 4, heads: 8


In [12]:
# Cell 8: Loss Function (Simple InfoNCE)
criterion = GFAv49Loss(
    label_smoothing=0.05,
    cosine_weight=0.3,  # Helps early training
)

# Scaler for FP16
scaler = GradScaler('cuda')

print("Loss function: GFAv49Loss (Simple InfoNCE)")
print(f"  Label smoothing: {criterion.label_smoothing}")
print(f"  Cosine weight: {criterion.cosine_weight}")

Loss function: GFAv49Loss (Simple InfoNCE)
  Label smoothing: 0.05
  Cosine weight: 0.3


In [13]:
# Cell 9: Verify Stage 0 Forward Pass
print("Verifying Stage 0 forward pass (BRep-PC only)...")

model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast('cuda'):
        outputs = model.forward_stage0(test_batch)

print("\nStage 0 Outputs:")
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        nan_count = v.isnan().sum().item()
        status = "NaN!" if nan_count > 0 else "OK"
        print(f"  {k}: {v.shape} [{status}]")
    else:
        print(f"  {k}: {v}")

# Test loss
loss, loss_dict = criterion(outputs, stage=0)
print("\nLoss components:")
for k, v in loss_dict.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.item():.4f}")

# Initial margin
print(f"\nInitial margin (should be near 0): {loss_dict['margin'].item():.4f}")

Verifying Stage 0 forward pass (BRep-PC only)...

Stage 0 Outputs:
  z_brep: torch.Size([512, 128]) [OK]
  z_pc: torch.Size([512, 128]) [OK]
  z_brep_pooled: torch.Size([512, 256]) [OK]
  z_pc_pooled: torch.Size([512, 256]) [OK]
  tau: torch.Size([]) [OK]

Loss components:
  infonce: 6.3527
  cosine: 0.9598
  total: 6.6407
  margin: 0.0032
  pos_sim: 0.0776
  neg_sim: 0.0744

Initial margin (should be near 0): 0.0032


## Stage 0: Anchor BRep to PC

**Goal:** Make BRep encoder produce meaningful features by aligning to pre-trained PC encoder.

**Key Metric:** margin (should reach > 0.3 by epoch 8)

**Expected Timeline:**
- Epoch 3: margin > 0.1
- Epoch 8: margin > 0.3
- If margin ~0: problem with data pipeline or BRep features
# Cell 10: Stage 0 Training
print("\n" + "="*70)
print("STAGE 0: Anchor BRep to PC")
print("="*70)

# Freeze PC encoder (anchor target)
model.freeze_pc_encoder()

# Optimizer with warmup scheduler
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=STAGE0_LR, 
    weight_decay=WEIGHT_DECAY
)
scheduler = get_warmup_cosine_scheduler(
    optimizer,
    warmup_epochs=STAGE0_WARMUP,
    total_epochs=STAGE0_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
)

stage0_history = {
    'loss': [], 'margin': [], 'pos_sim': [], 'neg_sim': [], 'lr': []
}

global_epoch = 0

for epoch in range(1, STAGE0_EPOCHS + 1):
    global_epoch += 1
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {'margin': [], 'pos_sim': [], 'neg_sim': []}
    
    pbar = tqdm(train_loader, desc=f"Stage 0, Epoch {epoch}/{STAGE0_EPOCHS}")
    
    for batch in pbar:
        batch = remap_batch(batch)
        
        with autocast('cuda'):
            outputs = model.forward_stage0(batch)
            loss, loss_dict = criterion(outputs, stage=0)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_loss += loss_dict['total'].item()
        epoch_metrics['margin'].append(loss_dict['margin'].item())
        epoch_metrics['pos_sim'].append(loss_dict['pos_sim'].item())
        epoch_metrics['neg_sim'].append(loss_dict['neg_sim'].item())
        
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.3f}",
            'margin': f"{loss_dict['margin'].item():.3f}",
        })
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_margin = np.mean(epoch_metrics['margin'])
    current_lr = scheduler.get_last_lr()[0]
    
    stage0_history['loss'].append(avg_loss)
    stage0_history['margin'].append(avg_margin)
    stage0_history['pos_sim'].append(np.mean(epoch_metrics['pos_sim']))
    stage0_history['neg_sim'].append(np.mean(epoch_metrics['neg_sim']))
    stage0_history['lr'].append(current_lr)
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Margin={avg_margin:.4f}, "
          f"Pos={np.mean(epoch_metrics['pos_sim']):.3f}, Neg={np.mean(epoch_metrics['neg_sim']):.3f}")
    
    # Early warning
    if epoch >= 3 and avg_margin < 0.05:
        print("  WARNING: Margin too low! Check data pipeline.")

# Save Stage 0 checkpoint
torch.save({
    'epoch': STAGE0_EPOCHS,
    'global_epoch': global_epoch,
    'model_state_dict': model.state_dict(),
    'history': stage0_history,
    'config': config,
}, OUTPUT_DIR / 'checkpoint_stage0.pt')

print(f"\nStage 0 complete! Final margin: {stage0_history['margin'][-1]:.4f}")

In [ ]:
# Cell 10b: Load Stage 0 Checkpoint (ALTERNATIVE to Cell 10)
# Run this cell INSTEAD of Cell 10 if you already trained Stage 0

print("\n" + "="*70)
print("Loading Stage 0 checkpoint...")
print("="*70)

STAGE0_CHECKPOINT = OUTPUT_DIR / 'checkpoint_stage0.pt'

if not STAGE0_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Stage 0 checkpoint not found at {STAGE0_CHECKPOINT}\n"
        f"Run Cell 10 to train Stage 0 first."
    )

checkpoint = torch.load(STAGE0_CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
stage0_history = checkpoint['history']
global_epoch = checkpoint.get('global_epoch', STAGE0_EPOCHS)

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"  Final margin: {stage0_history['margin'][-1]:.4f}")
print(f"  Final loss: {stage0_history['loss'][-1]:.4f}")

print("\nStage 0 checkpoint loaded successfully!")

## Stage 1: Full 3-Way Alignment

**Goal:** Align text, BRep, and PC in shared space using 3-way InfoNCE.

**Key Metrics:**
- margin (for all 3 pairs)
- R@1, R@5, R@10 (evaluated every 5 epochs)

**Expected Timeline:**
- Epoch 5: R@1 > 5%
- Epoch 10: R@1 > 20%, margin > 0.2
- Epoch 20: R@1 > 40-60%, margin > 0.4

In [15]:
# Cell 11: Verify Stage 1 Forward Pass
print("Verifying Stage 1 forward pass (full 3-way)...")

# Unfreeze PC encoder
model.unfreeze_pc_encoder()

model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast('cuda'):
        outputs = model(test_batch, stage=1)

print("\nStage 1 Outputs:")
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        nan_count = v.isnan().sum().item()
        status = "NaN!" if nan_count > 0 else "OK"
        print(f"  {k}: {v.shape} [{status}]")
    else:
        print(f"  {k}: {v}")

# Test loss
loss, loss_dict = criterion(outputs, stage=1)
print("\nLoss components:")
for k, v in loss_dict.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.item():.4f}")

Verifying Stage 1 forward pass (full 3-way)...
PC encoder unfrozen

Stage 1 Outputs:
  z_text: torch.Size([512, 128]) [OK]
  z_brep: torch.Size([512, 128]) [OK]
  z_pc: torch.Size([512, 128]) [OK]
  z_text_pooled: torch.Size([512, 256]) [OK]
  z_brep_pooled: torch.Size([512, 256]) [OK]
  z_pc_pooled: torch.Size([512, 256]) [OK]
  tau: torch.Size([]) [OK]

Loss components:
  infonce_t2b: 6.5827
  infonce_t2p: 6.5074
  infonce_b2p: 4.8581
  infonce: 5.9827
  total: 5.9827
  margin_tb: 0.0003
  pos_sim_tb: 0.0036
  neg_sim_tb: 0.0033
  margin_tp: -0.0002
  margin_bp: 0.1335
  margin: 0.0445


In [16]:
# Cell 12: Stage 1 Training
print("\n" + "="*70)
print("STAGE 1: Full 3-Way Alignment")
print("="*70)

# Optimizer with warmup scheduler
optimizer = AdamW(model.parameters(), lr=STAGE1_LR, weight_decay=WEIGHT_DECAY)
scheduler = get_warmup_cosine_scheduler(
    optimizer,
    warmup_epochs=STAGE1_WARMUP,
    total_epochs=STAGE1_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
)

stage1_history = {
    'loss': [], 'margin': [], 'margin_tb': [], 'margin_tp': [], 'margin_bp': [],
    'infonce': [], 'lr': [],
}
eval_history = {'epoch': [], 'r1_t2b': [], 'r5_t2b': [], 'r10_t2b': []}

EVAL_EVERY = 5  # Evaluate retrieval every 5 epochs

for epoch in range(1, STAGE1_EPOCHS + 1):
    global_epoch += 1
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {
        'margin': [], 'margin_tb': [], 'margin_tp': [], 'margin_bp': [], 'infonce': []
    }
    
    pbar = tqdm(train_loader, desc=f"Stage 1, Epoch {epoch}/{STAGE1_EPOCHS}")
    
    for batch in pbar:
        batch = remap_batch(batch)
        
        with autocast('cuda'):
            outputs = model(batch, stage=1)
            loss, loss_dict = criterion(outputs, stage=1)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_loss += loss_dict['total'].item()
        epoch_metrics['margin'].append(loss_dict['margin'].item())
        epoch_metrics['margin_tb'].append(loss_dict['margin_tb'].item())
        epoch_metrics['margin_tp'].append(loss_dict['margin_tp'].item())
        epoch_metrics['margin_bp'].append(loss_dict['margin_bp'].item())
        epoch_metrics['infonce'].append(loss_dict['infonce'].item())
        
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.3f}",
            'margin': f"{loss_dict['margin'].item():.3f}",
        })
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_margin = np.mean(epoch_metrics['margin'])
    current_lr = scheduler.get_last_lr()[0]
    
    stage1_history['loss'].append(avg_loss)
    stage1_history['margin'].append(avg_margin)
    stage1_history['margin_tb'].append(np.mean(epoch_metrics['margin_tb']))
    stage1_history['margin_tp'].append(np.mean(epoch_metrics['margin_tp']))
    stage1_history['margin_bp'].append(np.mean(epoch_metrics['margin_bp']))
    stage1_history['infonce'].append(np.mean(epoch_metrics['infonce']))
    stage1_history['lr'].append(current_lr)
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Margin={avg_margin:.4f}, "
          f"TB={np.mean(epoch_metrics['margin_tb']):.3f}, "
          f"TP={np.mean(epoch_metrics['margin_tp']):.3f}, "
          f"BP={np.mean(epoch_metrics['margin_bp']):.3f}")
    
    # Periodic evaluation
    if epoch % EVAL_EVERY == 0 or epoch == STAGE1_EPOCHS:
        print("  Evaluating retrieval...")
        model.eval()
        all_z_text, all_z_brep = [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="  Encoding", leave=False):
                batch = remap_batch(batch)
                with autocast('cuda'):
                    outputs = model(batch, stage=1)
                all_z_text.append(outputs['z_text'].cpu())
                all_z_brep.append(outputs['z_brep'].cpu())
        
        z_text = torch.cat(all_z_text, dim=0)
        z_brep = torch.cat(all_z_brep, dim=0)
        
        metrics = compute_retrieval_metrics(z_text, z_brep)
        eval_history['epoch'].append(epoch)
        eval_history['r1_t2b'].append(metrics['R@1'])
        eval_history['r5_t2b'].append(metrics['R@5'])
        eval_history['r10_t2b'].append(metrics['R@10'])
        
        print(f"  Text→BRep: R@1={metrics['R@1']:.1f}%, R@5={metrics['R@5']:.1f}%, R@10={metrics['R@10']:.1f}%")
        model.train()

# Save Stage 1 checkpoint
torch.save({
    'epoch': STAGE0_EPOCHS + STAGE1_EPOCHS,
    'global_epoch': global_epoch,
    'model_state_dict': model.state_dict(),
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'eval_history': eval_history,
    'config': config,
}, OUTPUT_DIR / 'checkpoint_stage1.pt')

print(f"\nStage 1 complete! Final margin: {stage1_history['margin'][-1]:.4f}")
if eval_history['r1_t2b']:
    print(f"Final R@1: {eval_history['r1_t2b'][-1]:.1f}%")


STAGE 1: Full 3-Way Alignment


Stage 1, Epoch 1/20:   0%|          | 0/216 [00:00<?, ?it/s]

Epoch 1: Loss=5.7544, Margin=0.0646, TB=0.001, TP=0.000, BP=0.192


Stage 1, Epoch 2/20:   0%|          | 0/216 [00:00<?, ?it/s]

Epoch 2: Loss=5.5270, Margin=0.0880, TB=0.011, TP=0.002, BP=0.251


...

Epoch 19: Loss=4.9933, Margin=0.1437, TB=0.036, TP=0.013, BP=0.382


Stage 1, Epoch 20/20:   0%|          | 0/216 [00:00<?, ?it/s]

Epoch 20: Loss=4.9903, Margin=0.1441, TB=0.036, TP=0.014, BP=0.382
  Evaluating retrieval...


  Encoding:   0%|          | 0/33 [00:00<?, ?it/s]

  Text→BRep: R@1=0.0%, R@5=0.1%, R@10=0.1%

Stage 1 complete! Final margin: 0.1441
Final R@1: 0.0%


## Stage 2: Fine-Tuning with Hard Negatives (Optional)

**Goal:** Improve fine-grained discrimination using hard negative mining.

Only run this if Stage 1 performance is reasonable (R@1 > 30%).

In [17]:
# Cell 13: Mine Hard Negatives
print("\n" + "="*70)
print("Mining hard negatives before Stage 2...")
print("="*70)

hard_negatives = mine_hard_negatives_simple(
    model, train_loader, device,
    top_k=5, max_batches=50,
    remap_fn=remap_batch
)
print(f"Mined {len(hard_negatives)} hard negative sets")


Mining hard negatives before Stage 2...
Mining hard negatives by embedding similarity...
Mined hard negatives for 25600 samples
Mined 25600 hard negative sets


In [18]:
# Cell 14: Stage 2 Training
print("\n" + "="*70)
print("STAGE 2: Fine-Tuning with Hard Negatives")
print("="*70)

# Optimizer with lower LR
optimizer = AdamW(model.parameters(), lr=STAGE2_LR, weight_decay=WEIGHT_DECAY)
scheduler = get_warmup_cosine_scheduler(
    optimizer,
    warmup_epochs=STAGE2_WARMUP,
    total_epochs=STAGE2_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
)

stage2_history = {
    'loss': [], 'margin': [], 'margin_tb': [], 'hard_neg': [], 'lr': []
}

best_r1 = eval_history['r1_t2b'][-1] if eval_history['r1_t2b'] else 0

for epoch in range(1, STAGE2_EPOCHS + 1):
    global_epoch += 1
    
    # Re-mine hard negatives every 2 epochs
    if epoch > 1 and epoch % 2 == 0:
        print("Re-mining hard negatives...")
        hard_negatives = mine_hard_negatives_simple(
            model, train_loader, device,
            top_k=5, max_batches=50,
            remap_fn=remap_batch
        )
    
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {'margin': [], 'margin_tb': [], 'hard_neg': []}
    
    pbar = tqdm(train_loader, desc=f"Stage 2, Epoch {epoch}/{STAGE2_EPOCHS}")
    
    for batch_idx, batch in enumerate(pbar):
        batch = remap_batch(batch)
        batch_size = batch['face_features'].shape[0]
        start_idx = batch_idx * batch_size
        batch_hard_negs = [
            hard_negatives.get(start_idx + i) for i in range(batch_size)
        ]
        
        with autocast('cuda'):
            outputs = model(batch, stage=1)  # Same forward as stage 1
            loss, loss_dict = criterion(
                outputs, stage=2,
                hard_negatives=batch_hard_negs,
                hard_neg_boost=1.5
            )
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_loss += loss_dict['total'].item()
        epoch_metrics['margin'].append(loss_dict['margin'].item())
        epoch_metrics['margin_tb'].append(loss_dict['margin_tb'].item())
        epoch_metrics['hard_neg'].append(loss_dict['hard_neg'].item())
        
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.3f}",
            'margin': f"{loss_dict['margin'].item():.3f}",
        })
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_margin = np.mean(epoch_metrics['margin'])
    current_lr = scheduler.get_last_lr()[0]
    
    stage2_history['loss'].append(avg_loss)
    stage2_history['margin'].append(avg_margin)
    stage2_history['margin_tb'].append(np.mean(epoch_metrics['margin_tb']))
    stage2_history['hard_neg'].append(np.mean(epoch_metrics['hard_neg']))
    stage2_history['lr'].append(current_lr)
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Margin={avg_margin:.4f}")
    
    # Final evaluation
    if epoch == STAGE2_EPOCHS:
        print("  Final evaluation...")
        model.eval()
        all_z_text, all_z_brep = [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="  Encoding", leave=False):
                batch = remap_batch(batch)
                with autocast('cuda'):
                    outputs = model(batch, stage=1)
                all_z_text.append(outputs['z_text'].cpu())
                all_z_brep.append(outputs['z_brep'].cpu())
        
        z_text = torch.cat(all_z_text, dim=0)
        z_brep = torch.cat(all_z_brep, dim=0)
        
        metrics = compute_retrieval_metrics(z_text, z_brep)
        print(f"  Final Text→BRep: R@1={metrics['R@1']:.1f}%, R@5={metrics['R@5']:.1f}%, R@10={metrics['R@10']:.1f}%")
        
        if metrics['R@1'] > best_r1:
            best_r1 = metrics['R@1']
            torch.save({
                'epoch': global_epoch,
                'model_state_dict': model.state_dict(),
                'best_r1': best_r1,
                'config': config,
            }, OUTPUT_DIR / 'checkpoint_best.pt')
            print(f"  New best R@1: {best_r1:.1f}%")

# Save Stage 2 checkpoint
torch.save({
    'epoch': STAGE0_EPOCHS + STAGE1_EPOCHS + STAGE2_EPOCHS,
    'global_epoch': global_epoch,
    'model_state_dict': model.state_dict(),
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'stage2_history': stage2_history,
    'eval_history': eval_history,
    'config': config,
}, OUTPUT_DIR / 'checkpoint_stage2.pt')

print("\n" + "="*70)
print("Training Complete!")
print(f"Best R@1: {best_r1:.1f}%")
print("="*70)


STAGE 2: Fine-Tuning with Hard Negatives


Stage 2, Epoch 1/5:   0%|          | 0/216 [00:00<?, ?it/s]

Epoch 1: Loss=6.8023, Margin=0.1407
Re-mining hard negatives...
Mining hard negatives by embedding similarity...
Mined hard negatives for 25600 samples


Stage 2, Epoch 2/5:   0%|          | 0/216 [00:00<?, ?it/s]

Epoch 2: Loss=6.7852, Margin=0.1421


...

Epoch 4: Loss=6.7532, Margin=0.1446


Stage 2, Epoch 5/5:   0%|          | 0/216 [00:00<?, ?it/s]

Epoch 5: Loss=6.7412, Margin=0.1457
  Final evaluation...


  Encoding:   0%|          | 0/33 [00:00<?, ?it/s]

  Final Text→BRep: R@1=0.0%, R@5=0.0%, R@10=0.1%

Training Complete!
Best R@1: 0.0%


In [ ]:
# Cell 15: Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Stage 0: Margin
ax = axes[0, 0]
ax.plot(stage0_history['margin'], label='Margin')
ax.axhline(y=0.3, color='g', linestyle='--', alpha=0.5, label='Target (0.3)')
ax.set_xlabel('Epoch')
ax.set_title('Stage 0: BRep-PC Margin')
ax.legend()
ax.grid(True, alpha=0.3)

# Stage 1: Margin
ax = axes[0, 1]
ax.plot(stage1_history['margin'], label='Avg Margin')
ax.plot(stage1_history['margin_tb'], label='Text-BRep', alpha=0.7)
ax.plot(stage1_history['margin_tp'], label='Text-PC', alpha=0.7)
ax.plot(stage1_history['margin_bp'], label='BRep-PC', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_title('Stage 1: 3-Way Margin')
ax.legend()
ax.grid(True, alpha=0.3)

# Retrieval
ax = axes[0, 2]
if eval_history['epoch']:
    ax.plot(eval_history['epoch'], eval_history['r1_t2b'], 'o-', label='R@1')
    ax.plot(eval_history['epoch'], eval_history['r5_t2b'], 's-', label='R@5')
    ax.plot(eval_history['epoch'], eval_history['r10_t2b'], '^-', label='R@10')
ax.set_xlabel('Epoch')
ax.set_ylabel('Recall (%)')
ax.set_title('Text→BRep Retrieval')
ax.legend()
ax.grid(True, alpha=0.3)

# Loss
ax = axes[1, 0]
all_loss = stage0_history['loss'] + stage1_history['loss']
if 'stage2_history' in dir() and stage2_history['loss']:
    all_loss += stage2_history['loss']
ax.plot(all_loss)
ax.axvline(x=STAGE0_EPOCHS, color='r', linestyle='--', alpha=0.3, label='Stage 1 start')
ax.axvline(x=STAGE0_EPOCHS + STAGE1_EPOCHS, color='r', linestyle='--', alpha=0.3, label='Stage 2 start')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.grid(True, alpha=0.3)

# Stage 0: Similarity
ax = axes[1, 1]
ax.plot(stage0_history['pos_sim'], label='Positive')
ax.plot(stage0_history['neg_sim'], label='Negative')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cosine Similarity')
ax.set_title('Stage 0: Similarity')
ax.legend()
ax.grid(True, alpha=0.3)

# Summary
axes[1, 2].axis('off')
final_r1 = eval_history['r1_t2b'][-1] if eval_history['r1_t2b'] else 0
summary_text = (
    f"GFA v4.9 Training Summary\n"
    f"(No Codebook - Direct Contrastive)\n\n"
    f"Model: d={config.d}, {model.count_parameters():,} params\n\n"
    f"Stage 0 (Anchor): {STAGE0_EPOCHS} epochs\n"
    f"  Margin: {stage0_history['margin'][-1]:.4f}\n\n"
    f"Stage 1 (Align): {STAGE1_EPOCHS} epochs\n"
    f"  Margin: {stage1_history['margin'][-1]:.4f}\n"
    f"  R@1: {final_r1:.1f}%\n"
)
if 'stage2_history' in dir() and stage2_history['loss']:
    summary_text += (
        f"\nStage 2 (Fine-tune): {STAGE2_EPOCHS} epochs\n"
        f"  Margin: {stage2_history['margin'][-1]:.4f}\n"
        f"  Best R@1: {best_r1:.1f}%"
    )
axes[1, 2].text(0.5, 0.5, summary_text, ha='center', va='center',
                fontsize=11, family='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

In [ ]:
# Cell 16: Save Final Model
torch.save({
    'config': config,
    'model_state_dict': model.state_dict(),
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'stage2_history': stage2_history if 'stage2_history' in dir() else None,
    'eval_history': eval_history,
}, OUTPUT_DIR / 'gfa_v4_9_final.pt')

print(f"Final model saved to {OUTPUT_DIR / 'gfa_v4_9_final.pt'}")
print(f"Model parameters: {model.count_parameters():,}")

In [19]:
# Cell 17: Full Retrieval Evaluation
print("\nFull retrieval evaluation...")

model.eval()
all_z_text = []
all_z_brep = []
all_z_pc = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Encoding"):
        batch = remap_batch(batch)
        with autocast('cuda'):
            outputs = model(batch, stage=1)
        
        all_z_text.append(outputs['z_text'].cpu())
        all_z_brep.append(outputs['z_brep'].cpu())
        all_z_pc.append(outputs['z_pc'].cpu())

z_text = torch.cat(all_z_text, dim=0)
z_brep = torch.cat(all_z_brep, dim=0)
z_pc = torch.cat(all_z_pc, dim=0)

# Normalize
z_text = F.normalize(z_text, dim=-1)
z_brep = F.normalize(z_brep, dim=-1)
z_pc = F.normalize(z_pc, dim=-1)

N = z_text.shape[0]
print(f"\nEvaluation set: {N} samples")

# Text -> BRep
metrics_t2b = compute_retrieval_metrics(z_text, z_brep)

# Text -> PC
metrics_t2p = compute_retrieval_metrics(z_text, z_pc)

# BRep -> PC
metrics_b2p = compute_retrieval_metrics(z_brep, z_pc)

# Contrastive quality
quality_tb = compute_contrastive_quality(z_text, z_brep)
quality_tp = compute_contrastive_quality(z_text, z_pc)
quality_bp = compute_contrastive_quality(z_brep, z_pc)

print("\n" + "="*50)
print("Retrieval Results (v4.9):")
print("="*50)
print(f"Text → BRep: R@1={metrics_t2b['R@1']:.1f}%, R@5={metrics_t2b['R@5']:.1f}%, R@10={metrics_t2b['R@10']:.1f}%")
print(f"Text → PC:   R@1={metrics_t2p['R@1']:.1f}%, R@5={metrics_t2p['R@5']:.1f}%")
print(f"BRep → PC:   R@1={metrics_b2p['R@1']:.1f}%, R@5={metrics_b2p['R@5']:.1f}%")
print("="*50)
print("\nContrastive Quality (margin = pos_sim - neg_sim):")
print(f"  Text-BRep: margin={quality_tb['margin']:.4f}")
print(f"  Text-PC:   margin={quality_tp['margin']:.4f}")
print(f"  BRep-PC:   margin={quality_bp['margin']:.4f}")
print("="*50)


Full retrieval evaluation...


Encoding:   0%|          | 0/33 [00:00<?, ?it/s]


Evaluation set: 16535 samples

Retrieval Results (v4.9):
Text → BRep: R@1=0.0%, R@5=0.0%, R@10=0.1%
Text → PC:   R@1=0.0%, R@5=0.1%
BRep → PC:   R@1=0.0%, R@5=0.0%

Contrastive Quality (margin = pos_sim - neg_sim):
  Text-BRep: margin=-0.0002
  Text-PC:   margin=0.0074
  BRep-PC:   margin=0.0117
